In [1]:
!pip install -q "transformers>=4.46,<4.50" "torch>=2.4" peft trl bitsandbytes accelerate datasets
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
torch: 2.4.1+cu124
transformers: 4.46.3


In [2]:
import gc, torch

# 커널에 남은 모델 변수 다 삭제
for name in ['trainer','model','base','_ours','peft_model']:
    if name in globals():
        try: del globals()[name]
        except: pass

gc.collect()
torch.cuda.empty_cache()
print("torch 할당:", round(torch.cuda.memory_allocated()/1e9,1), "GB")
print("여유:", round(torch.cuda.mem_get_info()[0]/1e9,1), "GB")

torch 할당: 0.0 GB
여유: 24.8 GB


In [2]:
import json
from collections import Counter

FILES = [
    "kanana_con_train.jsonl",
    "kanana_pep_train.jsonl",
    "kanana_rpt_train.jsonl",
]
OUT = "kanana_all_train.jsonl"

rows = []
for f in FILES:
    with open(f, encoding="utf-8") as fp:
        n = 0
        for line in fp:
            line = line.strip()
            if line:
                rows.append(line)   # 파싱 없이 줄 그대로
                n += 1
    print(f"{f}: {n}건")

with open(OUT, "w", encoding="utf-8") as fp:
    for line in rows:
        fp.write(line + "\n")

print(f"\n합계 {len(rows)}건 → {OUT}")

# 검증: 태스크 분포
tasks = [json.loads(l)["meta"]["task"] for l in rows]
print("태스크:", dict(Counter(tasks)))

kanana_con_train.jsonl: 200건
kanana_pep_train.jsonl: 201건
kanana_rpt_train.jsonl: 200건

합계 601건 → kanana_all_train.jsonl
태스크: {'CON': 200, 'PEP': 201, 'RPT': 200}


In [3]:
# =================
# JSONL / JSONL.GZ 자동 판별 + 태스크 분포 확인
# =================

import json
import gzip
from pathlib import Path
from collections import Counter

def smart_open(path):
    path = Path(path)

    # gzip 파일인지 앞 2바이트로 확인
    with open(path, "rb") as f:
        magic = f.read(2)

    if magic == b"\x1f\x8b":
        print(f"[gzip 감지] {path}")
        return gzip.open(path, "rt", encoding="utf-8")
    else:
        print(f"[일반 jsonl 감지] {path}")
        return open(path, "r", encoding="utf-8")


def check_jsonl(path):
    rows = []
    bad_lines = []
    task_counter = Counter()

    with smart_open(path) as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
                rows.append(obj)

                task = obj.get("meta", {}).get("task", "UNKNOWN")
                task_counter[task] += 1

            except Exception as e:
                bad_lines.append({
                    "line_no": line_no,
                    "error": str(e),
                    "preview": line[:300]
                })

    print("\n파일:", path)
    print("정상 파싱:", len(rows))
    print("깨진 줄:", len(bad_lines))
    print("깨진 줄 번호:", [x["line_no"] for x in bad_lines])
    print("태스크 분포:", dict(task_counter))

    if bad_lines:
        print("\n깨진 줄 미리보기:")
        for x in bad_lines[:5]:
            print("-" * 80)
            print("line:", x["line_no"])
            print("error:", x["error"])
            print("preview:", x["preview"])

    return rows, bad_lines


train_rows, train_bad = check_jsonl("kanana_all_train.jsonl")
valid_rows, valid_bad = check_jsonl("kanana_all_valid.jsonl")

[일반 jsonl 감지] kanana_all_train.jsonl

파일: kanana_all_train.jsonl
정상 파싱: 601
깨진 줄: 0
깨진 줄 번호: []
태스크 분포: {'CON': 200, 'PEP': 201, 'RPT': 200}
[일반 jsonl 감지] kanana_all_valid.jsonl

파일: kanana_all_valid.jsonl
정상 파싱: 44
깨진 줄: 0
깨진 줄 번호: []
태스크 분포: {'CON': 15, 'PEP': 14, 'RPT': 15}


In [12]:
import os
for f in ['kanana_all_train.jsonl', 'kanana_all_valid.jsonl']:
    if os.path.exists(f):
        rows = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
        print(f"{f}: {len(rows)}건 ✓")
    else:
        print(f"{f}: ❌ 없음 — 업로드 필요")

kanana_all_train.jsonl: 601건 ✓
kanana_all_valid.jsonl: 44건 ✓


In [3]:
# =================
# 데이터 로드
# =================

from datasets import load_dataset

train_ds = load_dataset("json", data_files="kanana_all_train.jsonl", split="train")
valid_ds = load_dataset("json", data_files="kanana_all_valid.jsonl", split="train")
print(f"train {len(train_ds)} / valid {len(valid_ds)}")
print("샘플 messages 역할:", [m['role'] for m in train_ds[0]['messages']])

train 601 / valid 44
샘플 messages 역할: ['system', 'user', 'assistant']


In [4]:
# ===============================
# 모델 + 토크나이저 (4bit QLoRA)
# ===============================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "kakaocorp/kanana-1.5-8b-instruct-2505"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,228,864 || trainable%: 0.5196


In [5]:
# ==================================
# 학습 설정 (SFTConfig, 4090 최적화)
# ==================================

from trl import SFTConfig, SFTTrainer

OUT = "kanana_finetuned_200_re_v2"

cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    # 추가
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,

    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    bf16=True,
    bf16_full_eval=True,

    max_length=4608,

    packing=False,

    save_strategy="epoch", # 에폭마다 저장
    eval_strategy="epoch", # 에폭마다 평가
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=5,
    report_to="none",

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    optim="paged_adamw_8bit",
    max_grad_norm=1.0,
)

In [6]:
# ===================
# 트레이너 + 학습실행
# ===================

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Epoch,Training Loss,Validation Loss
0,0.333500,0.383409
2,0.183100,0.367988


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


TrainOutput(global_step=900, training_loss=0.3723409281174342, metrics={'train_runtime': 2604.5208, 'train_samples_per_second': 0.692, 'train_steps_per_second': 0.346, 'total_flos': 1.7545187371008e+17, 'train_loss': 0.3723409281174342})

In [7]:
# ==========
# 최종 저장
# ==========
trainer.save_model(OUT + "_best")
tokenizer.save_pretrained(OUT + "_best")
print("저장 완료:", OUT + "_best")

# ===========================
# 학습 로그(에폭별 loss) 확인
# ===========================
import pandas as pd
logs = trainer.state.log_history
evals = [l for l in logs if 'eval_loss' in l]
for e in evals:
    print(f"epoch {e['epoch']:.0f}: valid_loss {e['eval_loss']:.4f}")
print(f"\n최종 채택: valid_loss 최저 epoch")

저장 완료: kanana_finetuned_200_re_v2_best
epoch 1: valid_loss 0.3834
epoch 2: valid_loss 0.3605
epoch 3: valid_loss 0.3680

최종 채택: valid_loss 최저 epoch
